# Tropical Cyclones

## Data exploration and analysis

Based on Arnaud's work

In [ ]:
# Importing the libraries needed for data visualization
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns

In [ ]:
# Import the dataframe with a single header
df = pd.read_parquet("../data/ibtracs_single_header.parquet", engine="pyarrow")

In [ ]:
df.shape

In [ ]:
# working on a copy of the dataset to preserve an untouched source of the dfcon
df_ibtracs = df.copy()

Filling empty rows: from " " to NaN 

In [ ]:
pd.set_option("future.no_silent_downcasting", True)

for col in df_ibtracs.columns:
    df_ibtracs[col] = df_ibtracs[col].replace(" ", np.nan)
    try:
        df_ibtracs[col] = pd.to_numeric(df_ibtracs[col])
    except ValueError:
        pass

Removing all rows where the taget TD9636_STAGE is null

In [ ]:
df_ibtracs = df_ibtracs.dropna(subset=["TD9636_STAGE"])

SEASONS - The seasons are opposites depending on the hemisphere

Northern Hemisphere: Lat > 0
- Spring runs from March 1 to May 31;
- Summer runs from June 1 to August 31;
- Fall (autumn) runs from September 1 to November 30; and
- Winter runs from December 1 to February 28 (February 29 in a leap year).

Southern Hemisphere: Lat < 0
- Spring starts September 1 and ends November 30;
- Summer starts December 1 and ends February 28 (February 29 in a Leap Year);
- Fall (autumn) starts March 1 and ends May 31; and
- Winter starts June 1 and ends August 31.

In [ ]:
df_ibtracs["ISO_TIME"] = pd.to_datetime(df_ibtracs["ISO_TIME"])

In [ ]:
def get_season(date, latitude):
    month = date.month
    day = date.day

    if latitude >= 0:  # Northern Hemisphere
        match month:
            case 12 | 1 | 2:  # winter
                return 1
            case 3 | 4 | 5:  # spring
                return 2
            case 6 | 7 | 8:
                return 3
            case 9 | 10 | 11:  # falle
                return 4

    else:  # Southern Hemisphere
        match month:
            case 12 | 1 | 2:  # summer
                return 3
            case 3 | 4 | 5:  # fall
                return 4
            case 6 | 7 | 8:  # winter
                return 1
            case 9 | 10 | 11:  # spring
                return 2

In [ ]:
df_ibtracs["SEASON"] = df_ibtracs.apply(
    lambda row: get_season(row["ISO_TIME"], row["LAT"]), axis=1
)

WIND - source Arnaud

The goal here will be to try to gather as much data as possible to create a 'WIND' column that will collect data from all agencies. For optimization purposes, we will take the 'USA_WIND' column as a reference and then add the data from the other agencies

The unit of measurement used for WIND is knots, with measurements taken at different times depending on the agency:

- WMO_WIND : ?
- USA_WIND : 1-min.mean
- TOKYO_WIND : 10-min.mean
- CMA_WIND : 2-min.mean
- HKO_WIND : 10-min.mean
- NEWDELHI_WIND : 3-min.mean
- REUNION_WIND : 10-min.mean
- BOM_WIND : 10-min.mean
- WELLINGTON_WIND : 10-min.mean
- DS824_WIND : 1-min.mean
- TD9636_WIND : 1-min.mean
- NEUMANN_WIND : 1-min.mean

We need to ensure proper conversion to the correct unit of measurement. The documentation provides the values to apply for the conversion.

We will work with the 1-minute mean. To convert from max speed 10-minute mean to 1-minute mean: *1.12 (IBTrACS technical details p5, https://www.ncei.noaa.gov/sites/g/files/anmtlf171/files/2024-07/IBTrACS_version4r01_Technical_Details.pdf).

Therefore, we will not be able to work with columns that do not respect both units.

In [ ]:
# renaming USA_WIND as WIND
df_ibtracs.rename(columns={"USA_WIND": "WIND"}, inplace=True)

In [ ]:
df_ibtracs[
    ["BOM_WIND", "WELLINGTON_WIND", "REUNION_WIND", "HKO_WIND", "TOKYO_WIND"]
] = (
    df_ibtracs[
        [
            "BOM_WIND",
            "WELLINGTON_WIND",
            "REUNION_WIND",
            "HKO_WIND",
            "TOKYO_WIND",
        ]
    ]
    * 1.12
)

In [ ]:
wind_list = [
    "TD9636_WIND",
    "NEUMANN_WIND",
    "HKO_WIND",
    "TOKYO_WIND",
    "BOM_WIND",
    "WELLINGTON_WIND",
    "REUNION_WIND",
    "DS824_WIND",
]

In [ ]:
# if WIND is NaN we loop over the other columns to fill it
for column in wind_list:
    df_ibtracs["WIND"] = df_ibtracs["WIND"].fillna(value=df_ibtracs[column])

In [ ]:
df_ibtracs["WIND"].isnull().sum()

In [ ]:
# Drop rows where WIND is still null
df_ibtracs = df_ibtracs.dropna(subset=["WIND"])

For PRESSURE

same filler logic

In [ ]:
# renaming TOKYO_PRES as PRESSURE
df_ibtracs.rename(columns={"TOKYO_PRES": "PRESSURE"}, inplace=True)

In [ ]:
pressure_list = [
    "CMA_PRES",
    "WMO_PRES",
    "HKO_PRES",
    "BOM_PRES",
    "WELLINGTON_PRES",
    "REUNION_PRES",
    "NEUMANN_PRES",
    "USA_PRES",
    "BOM_PRES_METHOD",
    "DS824_PRES",
    "NEWDELHI_PRES",
]

In [ ]:
# if PRESSURE is NaN we loop over the other columns to fill it
for column in pressure_list:
    df_ibtracs["PRESSURE"] = df_ibtracs["PRESSURE"].fillna(
        value=df_ibtracs[column]
    )

In [ ]:
df_ibtracs["PRESSURE"].isnull().sum()

In [ ]:
# Drop rows where PRESSURE is still null
df_ibtracs = df_ibtracs.dropna(subset=["PRESSURE"])

## SELECTION

After cleaning the dataset:
- filling rows when possible 
- merging data when redundant
- dropping columns or rows when there is not enough data

We are left with the following columns:
- SEASON: replacing the current SEASON column, constructed from the ISO_TIME value to define the seasonal period of the year (spring, summer, autumn, winter)
- BASIN
- NATURE
- LAT
- LON
- WIND: column created by gathering data from the different agencies's _WIND columns when WMO_WIND is left empty. We take into account the differences in wind speed averaging periods depending on the agency to construct this column
- PRESSURE: same logic as for WIND
- DIST2LAND
- LANDFALL
- STORM_SPEED
- STORM_DIR

In [ ]:
df_ = df_ibtracs[
    [
        "SEASON",
        "BASIN",
        "NATURE",
        "LAT",
        "LON",
        "WIND",
        "PRESSURE",
        "DIST2LAND",
        "LANDFALL",
        "STORM_SPEED",
        "STORM_DIR",
        "TD9636_STAGE",
    ]
]

In [ ]:
df_.info()

In [ ]:
df_.isnull().sum().sort_values()

## Graphs

In [ ]:
df_encoded = pd.get_dummies(df_, columns=["BASIN", "NATURE"], drop_first=True)

In [ ]:
df_encoded.info()

Let's try and see if there is a correlation between the different columns of our dataframe

In [ ]:
# Compute correlation matrix
corr_matrix = df_encoded.corr()

# The correlation matrix combines two "triangles", they are redundant
# so we mask the upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
corr_matrix_masked = corr_matrix.mask(mask)

fig = go.Figure(
    data=go.Heatmap(
        z=corr_matrix_masked.values,  # Correlation values
        x=corr_matrix.columns,  # Column labels
        y=corr_matrix.index,  # Row labels
        colorscale="hsv",  # Color scheme
        zmin=-1,  # Minimum correlation value
        zmax=1,  # Maximum correlation value
        hoverongaps=False,
    )
)

fig.update_layout(title="Correlation Matrix", yaxis_autorange="reversed")

fig.show()

high correlation between WIND and PRESSURE
high correlation between DISTLAND and LANDFALL

In [ ]:
fig = px.histogram(df_, x="TD9636_STAGE", color="TD9636_STAGE", opacity=0.75)

fig.update_layout(
    title_text=f"Histogram of the TD9636_STAGE of the Cyclone",  # title of plot
    xaxis_title_text="TD9636_STAGE",  # x axis label
    yaxis_title_text="Count",  # y axis label
    # bargap=0.2, # gap between bars of adjacent location coordinates
    # bargroupgap=0.1 # gap between bars of the same location coordinates
)

fig.show()

### Histograms

For categorical columns

- SEASON
- BASIN
- NATURE

In [ ]:
def create_histogram(dataframe: pd.DataFrame, x_axis: str, color_column: str):
    fig = px.histogram(dataframe, x=x_axis, color=color_column, opacity=0.75)

    fig.update_layout(
        title_text=f"Histogram of the {x_axis} of the Cyclone",  # title of plot
        xaxis_title_text=x_axis,  # x axis label
        yaxis_title_text="Count",  # y axis label
        bargap=0.2,  # gap between bars of adjacent location coordinates
        bargroupgap=0.1,  # gap between bars of the same location coordinates
    )

    fig.show()

In [ ]:
create_histogram(df_, "SEASON", "TD9636_STAGE")

In [ ]:
create_histogram(df_, "BASIN", "TD9636_STAGE")

In [ ]:
create_histogram(df_, "NATURE", "TD9636_STAGE")

data augmentation for MX, NR, DS, ET ???

### Box plots

For numerical columns. It will show the distribution of the values across the different categories.

- WIND
- PRESSURE
- DIST2LAND
- LANDFALL
- STORM_SPEED
- STORM_DIR         

In [ ]:
def create_box_plot(dataframe: pd.DataFrame, feature: str):
    fig = go.Figure()

    for value in dataframe["TD9636_STAGE"].unique():
        fig.add_trace(
            go.Box(
                y=dataframe[dataframe["TD9636_STAGE"] == value][feature],
                name=f"Stage: {value}",
                boxmean="sd",
            )
        )

    fig.update_layout(
        title=f"Box plot of {feature} grouped by TD9636_STAGE",  # title of plot
        xaxis_title="Groups",  # x axis label
        yaxis_title=f"{feature} value",  # y axis label
        boxmode="group",  # groups together boxes of the same category
    )

    fig.show()

In [ ]:
create_box_plot(df_, "WIND")

In [ ]:
create_box_plot(df_, "PRESSURE")

In [ ]:
create_box_plot(df_, "DIST2LAND")

In [ ]:
create_box_plot(df_, "LANDFALL")

DIST2LAND and LANDFALL really equivalent 

In [ ]:
create_box_plot(df_, "STORM_SPEED")

In [ ]:
create_box_plot(df_, "STORM_DIR")

### Scatter plots

Shows the relation between columns

- WIND - PRESSURE
- DIST2LAND - LANDFALL
- STORM_SPEED - STORM_DIR  
- NATURE - LAT
- NATURE - LON   
- NATURE - SEASON    

In [ ]:
def create_scatter_plot(
    dataframe: pd.DataFrame, x_axis: str, y_axis: str, target: str
):
    fig = px.scatter(
        dataframe,
        x=x_axis,
        y=y_axis,
        color=target,
        title=f"Relation between {x_axis} and {y_axis}",
        labels={
            "SEASON": "season",
            "BASIN": "basin",
            "NATURE": "nature",
            "LAT": "latitude",
            "LON": "longitude",
            "WIND": "wind",
            "PRESSURE": "pressure",
            "DIST2LAND": "distance to land",
            "LANDFALL": "landfall",
            "STORM_SPEED": "storm speed",
            "STORM_DIR": "storm direction",
            "TD9636_STAGE": "TD9636 stage",
        },
        hover_data=[
            "SEASON",
            "BASIN",
            "NATURE",
            "LAT",
            "LON",
            "WIND",
            "PRESSURE",
            "DIST2LAND",
            "LANDFALL",
            "STORM_SPEED",
            "STORM_DIR",
            "TD9636_STAGE",
        ],
    )

    fig.update_layout(
        xaxis_title=x_axis,  # x axis label
        yaxis_title=y_axis,  # y axis label
        xaxis_tickformat="%S",
    )

    fig.show()

In [ ]:
create_scatter_plot(df_, "WIND", "PRESSURE", "TD9636_STAGE")

In [ ]:
create_scatter_plot(df_, "DIST2LAND", "LANDFALL", "TD9636_STAGE")

In [ ]:
create_scatter_plot(df_, "STORM_SPEED", "STORM_DIR", "TD9636_STAGE")

In [ ]:
create_scatter_plot(df_, "NATURE", "LAT", "TD9636_STAGE")

In [ ]:
create_scatter_plot(df_, "NATURE", "LON", "TD9636_STAGE")

In [ ]:
sns.jointplot(x="LON", y="WIND", hue="TD9636_STAGE", data=df_)

In [ ]:
sns.jointplot(x="LAT", y="WIND", hue="TD9636_STAGE", data=df_)

In [ ]:
sns.jointplot(x="LAT", y="STORM_DIR", hue="TD9636_STAGE", data=df_)

In [ ]:
sns.jointplot(x="STORM_DIR", y="LAT", hue="BASIN", data=df_)

In [ ]:
sns.jointplot(x="STORM_DIR", y="WIND", hue="BASIN", data=df_)

In [ ]:
sns.pairplot(df_)

In [ ]:
# ### works but heavy ####

# fig = px.scatter_geo(
#     df_,
#     lat="LAT",
#     lon="LON",
#     color="TD9636_STAGE",  # Color markers based on pressure
#     size="WIND",  # Size markers based on wind speed
#     hover_name="PRESSURE",
#     projection="natural earth"  # Map projection type
# )

# # Customize layout (optional)
# fig.update_layout(
#     title="Cyclone over the earth",
#     geo=dict(
#         showland=True,  # Show land on the map
#         landcolor="lightgray",  # Land color
#         showcountries=True,  # Show country borders
#         countrycolor="black"  # Country border color
#     )
# )

# # Show the figure
# fig.show()

We will now build models with these features and try to classify the storms depending on the target TD9636_STAGE. 

Categorical => OneHotEncoder or one dimension with different values (1, 2, 3, 4, etc.)
- SEASON oneHotEncoder (4)
- BASIN (7)
- NATURE (6)

Numeric => everything between 0 and 1
- LAT
- LON
- WIND 
- DIST2LAND
- STORM_SPEED
- STORM_DIR

## Side notes

Once the first model available we might try and improve the results by adding other columns and modifying further some of the actual columns.
Columns considered: 
- TRACK_TYPE
- IFLAG
- Try and use (merge or other methods) all the R30, R40, etc. content

Further modifications considered:
- Filling the empty TD9636_STAGE rows based on other columns that contain a classification of the storm (USA_SSHS, TOKYO_GRADE, CMA_CAT, HKO_CAT, KMA_CAT, NEWDELHI_GRADE, REUNION_TYPE, BOM_TYPE, NADI_CAT, DS824_STAGE, NEUMANN_CLASS, MLC_CLASS)
- Adding WIND rowsby converting NEWDELHI_WIND, CMA_WIND (and maybe WMO_WIND) as it was done for the others
- mergin BASIN and SUBBASIN to add more precise information 

- test  MONTH / SEASON
-with/without PRESSION (because approximations)